<a href="https://colab.research.google.com/github/LokmindTech/TMDB-p-1/blob/main/03_pandas_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
data_folder = Path("/content/drive/MyDrive/Cleaned data")

movies = pd.read_csv(data_folder / " movies.csv")
genres = pd.read_csv(data_folder / " genres.csv")
movie_genres = pd.read_csv(data_folder / " movie_genres.csv")
cast = pd.read_csv(data_folder / " cast.csv")
crew = pd.read_csv(data_folder / " crew.csv")
movie_keywords = pd.read_csv(data_folder / " movie_keywords.csv")

In [ ]:
print("Movies:", movies.shape)
print("Genres:", genres.shape)
print("Movie-genre links:", movie_genres.shape)
print("Cast:", cast.shape)
print("Crew:", crew.shape)
print("Keywords:", movie_keywords.shape)

Movies: (2503, 12)
Genres: (19, 2)
Movie-genre links: (6878, 2)
Cast: (24902, 6)
Crew: (7289, 6)
Keywords: (41681, 3)


# Pandas Analysis — 10 Questions

**1) List the total number of unique movies, genres, cast members and crew members collected across all tables.**

In [ ]:
print("Movies:", movies["movie_id"].nunique())
print("Genres:", genres["genre_id"].nunique())
print("Actors:", cast["person_id"].nunique())
print("Crew members:", crew["person_id"].nunique())

Movies: 2503
Genres: 19
Actors: 11300
Crew members: 2848


**2) List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.**

In [ ]:
result = movies[
    movies["budget"].isna() |
    movies["revenue"].isna() |
    (movies["budget"] == 0) |
    (movies["revenue"] == 0)
]
display(result[["title", "budget", "revenue"]])

# Zero means unknown in this dataset.
movies_analysis = movies.copy()
movies_analysis[["budget", "revenue"]] = movies_analysis[["budget", "revenue"]].replace(0, pd.NA)

,title,budget,revenue
324,Zack Snyder's Justice League,70000000.0,0.0
340,Bird Box,19800000.0,0.0
403,365 Days,0.0,9458590.0
468,To All the Boys I've Loved Before,0.0,0.0
554,Prey,65000000.0,0.0
...,...,...,...
2469,Day Shift,100000000.0,0.0
2471,A Serbian Film,0.0,0.0
2480,Chungking Express,160000.0,0.0
2481,I'm Thinking of Ending Things,0.0,0.0


**3) List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.**

In [ ]:
result = movies_analysis.dropna(subset=["budget", "revenue"]).copy()
result["profit"] = result["revenue"] - result["budget"]
result["ROI"] = (result["profit"] / result["budget"]) * 100

display(result[["title", "budget", "revenue", "profit", "ROI"]].head(10))

,title,budget,revenue,profit,ROI
0,Interstellar,165000000.0,746606706.0,581606706.0,352.488913
1,Inception,160000000.0,839030630.0,679030630.0,424.394144
2,The Avengers,220000000.0,1518815515.0,1298815515.0,590.370689
3,The Dark Knight,185000000.0,1004558444.0,819558444.0,443.004564
4,Avatar,237000000.0,2923706026.0,2686706026.0,1133.631235
5,Deadpool,58000000.0,782837347.0,724837347.0,1249.719564
6,Fight Club,63000000.0,100853753.0,37853753.0,60.085322
7,Avengers: Infinity War,300000000.0,2052415039.0,1752415039.0,584.138346
8,The Shawshank Redemption,25000000.0,28341469.0,3341469.0,13.365876
9,Pulp Fiction,8000000.0,213928762.0,205928762.0,2574.109525


**4) List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.**

In [ ]:
result = movies_analysis.dropna(subset=["budget", "revenue"]).copy()
result = result[result["budget"] >= 1000000]
result["profit"] = result["revenue"] - result["budget"]
result["ROI"] = (result["profit"] / result["budget"]) * 100
result = result.sort_values("ROI", ascending=False)

display(result[["title", "budget", "revenue", "ROI"]].head(15))

,title,budget,revenue,ROI
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,12400.2821
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,12127.242857
467,Rocky,1000000.0,117253345.0,11625.3345
1227,Gone with the Wind,4000000.0,402352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,9350.0
635,Cinderella,2900000.0,263600000.0,8989.655172
372,Saw,1200000.0,104045735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,8316.666667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,7493.403876


**5) List the number of movies released per year, and identify which year had the highest movie output in the dataset.**

In [ ]:
movies["year"] = movies["release_date"].dt.year

result = movies.groupby("year")["movie_id"].nunique().reset_index()
result.columns = ["year", "movie_count"]
result = result.sort_values("movie_count", ascending=False)

display(result)
print("Year with most movies:")
display(result.head(1))

,year,movie_count
78,2016,121
75,2013,110
79,2017,109
76,2014,105
80,2018,102
...,...,...
0,1921,1
11,1946,1
32,1970,1
34,1972,1


Year with most movies:


,year,movie_count
78,2016,121


**6) List all movies whose runtime is a statistical outlier compared to the rest of the dataset.**

In [ ]:
runtime = movies["runtime"].dropna()

Q1 = runtime.quantile(0.25)
Q3 = runtime.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

result = movies[(movies["runtime"] < lower) | (movies["runtime"] > upper)]
display(result[["title", "runtime"]])

,title,runtime
0,Interstellar,169
16,Avengers: Endgame,181
18,The Lord of the Rings: The Fellowship of the Ring,179
19,Titanic,194
20,The Lord of the Rings: The Return of the King,201
21,The Wolf of Wall Street,180
28,The Lord of the Rings: The Two Towers,179
38,The Godfather,175
76,The Hobbit: An Unexpected Journey,169
79,The Green Mile,189


**7) List each movie's primary genre (the first genre listed) and compare average popularity across primary genres.**

In [ ]:
first_genre = movie_genres.drop_duplicates("movie_id")

result = pd.merge(first_genre, genres, on="genre_id")
result = pd.merge(result, movies[["movie_id", "popularity"]], on="movie_id")

result = result.groupby("genre_name")["popularity"].mean().reset_index()
result = result.sort_values("popularity", ascending=False)

display(result)

,genre_name,popularity
14,Science Fiction,21.261836
11,Music,18.324340
2,Animation,16.811212
7,Family,16.710635
0,Action,16.385188
16,War,15.470317
10,Horror,14.851835
1,Adventure,14.785510
8,Fantasy,13.022920
12,Mystery,12.427131


**8)List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.**

In [ ]:
result = pd.merge(
    cast[["movie_id", "person_id", "actor_name"]].drop_duplicates(),
    movies[["movie_id", "vote_average"]],
    on="movie_id"
)

result = result.groupby(["person_id", "actor_name"]).agg(
    movie_count=("movie_id", "nunique"),
    average_rating=("vote_average", "mean")
).reset_index()

result = result.sort_values("movie_count", ascending=False)
display(result.head(10))

,person_id,actor_name,movie_count,average_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


**9) List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.**

In [ ]:
result = movies[movies["title"].duplicated(keep=False)]
result = result[["title", "release_date", "movie_id"]]
result = result.sort_values("title")

display(result)

,title,release_date,movie_id
878,A Nightmare on Elm Street,1984-11-09,377
1928,A Nightmare on Elm Street,2010-04-30,23437
261,Aladdin,1992-11-25,812
339,Aladdin,2019-05-22,420817
792,Alice in Wonderland,1951-08-09,12092
...,...,...,...
1737,The Thing,2011-10-12,60935
860,Total Recall,2012-08-02,64635
744,Total Recall,1990-06-01,861
252,Transformers,2007-06-27,1858


**10)List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high popularity with a low vote count, or vice versa.**

In [ ]:
high_popularity = movies["popularity"].quantile(0.75)
low_popularity = movies["popularity"].quantile(0.25)
high_votes = movies["vote_count"].quantile(0.75)
low_votes = movies["vote_count"].quantile(0.25)

result = movies[
    ((movies["popularity"] >= high_popularity) & (movies["vote_count"] <= low_votes)) |
    ((movies["popularity"] <= low_popularity) & (movies["vote_count"] >= high_votes))
]

display(result[["title", "popularity", "vote_count", "vote_average"]])